# Differential Equations — Session 6
## Section 2.3: Linear First-Order Equations

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to:

1. Put a first-order linear equation into standard form.
2. construct an integrating factor.
3. recognize the product derivative created by the integrating factor.
4. solve general and initial-value problems.
5. identify singular points and solution intervals.
6. solve a piecewise-forced linear equation.
7. recognize when a special function such as $\operatorname{erf}$ appears.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Suggested pacing

| Time | Topic |
|---:|---|
| 0–15 min | Standard form |
| 15–32 min | Deriving the integrating factor |
| 32–50 min | Worked IVP |
| 50–62 min | Singular points and intervals |
| 62–78 min | Piecewise forcing |
| 78–87 min | Error-function example |
| 87–90 min | Exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.special import erf
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=5, suppress=True)

def slope_field(f, xlim=(-3, 3), ylim=(-3, 3), density=21, title=None):
    x = np.linspace(*xlim, density)
    y = np.linspace(*ylim, density)
    X, Y = np.meshgrid(x, y)
    S = np.asarray(f(X, Y), dtype=float)
    S = np.nan_to_num(S, nan=0.0, posinf=20.0, neginf=-20.0)
    U = np.ones_like(S)
    length = np.sqrt(U**2 + S**2)
    plt.quiver(X, Y, U/length, S/length, angles="xy", pivot="mid")
    plt.xlim(*xlim)
    plt.ylim(*ylim)
    plt.xlabel("x")
    plt.ylabel("y")
    if title:
        plt.title(title)

def euler_method(f, x0, y0, h, n_steps):
    xs = np.empty(n_steps + 1)
    ys = np.empty(n_steps + 1)
    xs[0], ys[0] = x0, y0
    for n in range(n_steps):
        ys[n+1] = ys[n] + h*f(xs[n], ys[n])
        xs[n+1] = xs[n] + h
    return xs, ys

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 2.3-A — Linear first-order equation

A first-order equation is **linear in $y$** when it can be written as

$$
a_1(x)y'+a_0(x)y=g(x).
$$

Where $a_1(x)\ne0$, its standard form is

$$
y'+P(x)y=Q(x).
$$

A point where $a_1(x)=0$ is a **singular point** of the equation.

### Theorem 2.3-B — Integrating-factor formula

If $P$ and $Q$ are continuous on an interval $I$, define

$$
\mu(x)=e^{\int P(x)\,dx}.
$$

Then

$$
(\mu y)'=\mu Q,
$$

and the general solution on $I$ is

$$
y(x)=\frac{1}{\mu(x)}
\left[
C+\int \mu(x)Q(x)\,dx
\right].
$$

### Theorem 2.3-C — Existence and uniqueness for a linear IVP

If $P$ and $Q$ are continuous on an interval $I$ containing $x_0$, then

$$
y'+P(x)y=Q(x),
\qquad
y(x_0)=y_0
$$

has exactly one solution on the entire interval $I$.

### Structural interpretation

Every solution can be written as

$$
y=y_p+y_h,
$$

where $y_p$ is one particular solution and

$$
y_h=Ce^{-\int P(x)\,dx}
$$

is the homogeneous transient.

### Classroom Checkpoint — Integrating Factor

For

$$
y'+P(x)y=Q(x),
$$

what integrating factor should be used, and what product derivative does it create?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Standard form

A linear first-order equation is written as

$$
y'+P(x)y=Q(x).
$$

If the original leading coefficient is not $1$, divide first. Values where that coefficient is zero are singular points and may separate solution intervals.

## 2. Why the integrating factor works

We seek $\mu(x)$ such that

$$
\mu y'+\mu P y
$$

is the product derivative

$$
(\mu y)'=\mu y'+\mu' y.
$$

Thus we require

$$
\mu'=\mu P,
$$

so

$$
\mu(x)=e^{\int P(x)\,dx}.
$$

After multiplication,

$$
(\mu y)'=\mu Q,
$$

and therefore

$$
y=\frac{1}{\mu(x)}
\left(
\int \mu(x)Q(x)\,dx+C
\right).
$$

## 3. Worked example

Solve

$$
y'+2y=3x+1.
$$

The integrating factor is

$$
\mu=e^{2x}.
$$

Then

$$
(e^{2x}y)'=e^{2x}(3x+1).
$$

Integration gives

$$
y=\frac{3}{2}x-\frac14+Ce^{-2x}.
$$

In [ ]:
x = sp.symbols("x", real=True)
C = sp.symbols("C")
y = sp.Rational(3, 2)*x - sp.Rational(1, 4) + C*sp.exp(-2*x)
residual = sp.simplify(sp.diff(y, x) + 2*y - (3*x+1))
display(residual)

In [ ]:
x_vals = np.linspace(-1, 4, 600)
for C_value in [-4, -1, 0, 2, 5]:
    y_vals = 1.5*x_vals - 0.25 + C_value*np.exp(-2*x_vals)
    plt.plot(x_vals, y_vals, label=fr"$C={C_value}$")
plt.ylim(-8, 8)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Transient term plus long-term particular response")
plt.legend()
plt.show()

The term $Ce^{-2x}$ is a decaying transient. Every solution approaches the particular solution

$$
y_p=\frac32x-\frac14
$$

as $x\to\infty$.

## 4. Singular points

Solve

$$
xy'-2y=x^3.
$$

For $x\ne0$,

$$
y'-\frac{2}{x}y=x^2.
$$

The singular point $x=0$ divides the real line into $(-\infty,0)$ and $(0,\infty)$.

On either interval, an integrating factor is effectively $x^{-2}$, and

$$
(x^{-2}y)'=1.
$$

Thus

$$
y=x^3+Cx^2.
$$

In [ ]:
x_vals_left = np.linspace(-3, -0.05, 400)
x_vals_right = np.linspace(0.05, 3, 400)

for C_value in [-2, 0, 2]:
    plt.plot(x_vals_left, x_vals_left**3 + C_value*x_vals_left**2)
    plt.plot(x_vals_right, x_vals_right**3 + C_value*x_vals_right**2,
             label=fr"$C={C_value}$")

plt.axvline(0, linestyle="--", label="singular point")
plt.xlabel("x")
plt.ylabel("y")
plt.title(r"Solutions of $xy'-2y=x^3$ on intervals excluding $0$")
plt.legend()
plt.show()

## 5. Piecewise forcing

Consider

$$
y'+y=f(t),\qquad y(0)=0,
$$

where

$$
f(t)=
\begin{cases}
1,&0\le t<2,\\
3,&t\ge2.
\end{cases}
$$

For $0\le t<2$,

$$
y=1-e^{-t}.
$$

Continuity at $t=2$ gives, for $t\ge2$,

$$
y=3-\left(2+e^{-2}\right)e^{-(t-2)}.
$$

In [ ]:
t = np.linspace(0, 8, 800)
forcing = np.where(t < 2, 1.0, 3.0)
solution = np.where(
    t < 2,
    1-np.exp(-t),
    3-(2+np.exp(-2))*np.exp(-(t-2))
)

plt.plot(t, forcing, linestyle="--", label="input f(t)")
plt.plot(t, solution, linewidth=2, label="response y(t)")
plt.axvline(2, linestyle=":")
plt.xlabel("t")
plt.ylabel("value")
plt.title("Continuous state response to a discontinuous input")
plt.legend()
plt.show()

## 6. A nonelementary integral and the error function

Solve

$$
y'-2xy=1,\qquad y(0)=0.
$$

The integrating factor is

$$
\mu=e^{-x^2}.
$$

Therefore

$$
(e^{-x^2}y)'=e^{-x^2},
$$

and

$$
y=e^{x^2}\int_0^x e^{-t^2}\,dt
=\frac{\sqrt{\pi}}{2}e^{x^2}\operatorname{erf}(x).
$$

In [ ]:
x_vals = np.linspace(-2, 2, 500)
y_vals = np.sqrt(np.pi)/2*np.exp(x_vals**2)*erf(x_vals)

plt.plot(x_vals, y_vals, linewidth=2)
plt.scatter([0], [0], s=70)
plt.xlabel("x")
plt.ylabel("y")
plt.title(r"Solution involving $\operatorname{erf}(x)$")
plt.show()

## Interactive exploration — Transient decay rate

For

$$
y'+ay=3x+1,
$$

increasing $a>0$ changes both the particular response and the rate at which the homogeneous transient decays.

In [ ]:
def linear_response(a=2.0, C=3.0):
    x = np.linspace(0, 6, 600)

    alpha = 3/a
    beta = (1-alpha)/a
    yp = alpha*x + beta
    y = yp + C*np.exp(-a*x)

    plt.plot(x, y, linewidth=2, label="complete solution")
    plt.plot(x, yp, linestyle="--", label="particular response")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(fr"$y'+{a:.2f}y=3x+1$")
    plt.legend()
    plt.show()

    print("Transient e-folding time:", 1/a)

if WIDGETS_AVAILABLE:
    interact(
        linear_response,
        a=FloatSlider(min=0.25, max=5.0, step=0.25, value=2.0),
        C=FloatSlider(min=-6, max=6, step=0.5, value=3.0)
    )
else:
    linear_response()

## Classroom Checkpoint — Exit Check

For

$$
(1+x)y'+2y=x^2,
$$

state:

1. the standard form,
2. the singular point,
3. an integrating factor on $x>-1$.

> Pause here. Let students commit to an answer before running the next cell.